[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/quickstart/quickstart_langchain.ipynb)

# DecimalAI + LangChain Quickstart

**Instrument a LangChain agent with 2 lines of code.**

This notebook shows how to add DecimalAI tracing to any LangChain / LangGraph
application. Every LLM call, tool invocation, and chain step is captured automatically.

**Prerequisites:** an OpenAI API key (or any LangChain-supported LLM) for Steps 2-3 —
that is the LLM the agent runs on, and there is no agent run without it. A DecimalAI
key is **optional**: with one, the runs are traced to your dashboard; without one the
notebook initializes the SDK with tracing switched off and still runs, rather than
stopping at the second cell. Neither missing key raises — each cell says which one it
would need.

## Step 1 — Install & Configure

In [ ]:
# Install dependencies (the [langchain] extra brings the adapter deps;
# `langchain` provides create_agent, `langchain-openai` the LLM binding)
!pip install -q "decimalai[langchain]" langchain langchain-openai

In [ ]:
import os

import decimalai

# A key is OPTIONAL here. This cell looks for one, and if it doesn't find a real
# one it initializes the SDK in offline mode instead of raising: `enabled=False`
# is a real kill switch: no client, no network call, and the langchain=True flag
# is ignored, so no adapter is installed and nothing is captured. The LangChain
# agent below still runs — it just runs untraced.
#
# To send traces to your own dashboard, get a key at
# https://app.decimal.ai/settings (Settings -> General -> API keys) and either:
#   * add it as a Colab secret named DECIMAL_API_KEY (the key icon, left sidebar),
#   * export DECIMAL_API_KEY before launching Jupyter, or
#   * flip ASK_FOR_KEY to True below and paste it into the hidden prompt.
ASK_FOR_KEY = False

# The literal this notebook used to assign to DECIMAL_API_KEY. Treated as "no key"
# rather than passed through, so a reader who pastes the snippet from the docs and
# forgets to edit it lands in offline mode instead of on a 401.
PLACEHOLDER = "dai_sk_..."


def find_key(*names) -> str:
    """First real key among env vars, Colab secrets, and (opt-in) a prompt."""
    for name in names:
        value = (os.environ.get(name) or "").strip()
        if value and value != PLACEHOLDER:
            return value

    try:
        from google.colab import userdata  # only exists inside Colab
    except ImportError:
        pass
    else:
        for name in names:
            try:
                value = (userdata.get(name) or "").strip()
            except Exception:
                continue  # secret not set, or notebook denied access to it
            if value and value != PLACEHOLDER:
                return value

    if ASK_FOR_KEY:
        import getpass
        # getpass, never input(): a key typed into a cell is a key in the
        # browser history and in the .ipynb you later share.
        return getpass.getpass(f"{names[0]} (input is hidden): ").strip()

    return ""


# DECIMALAI_API_KEY is the alias the CLI also accepts; init() reads both.
API_KEY = find_key("DECIMAL_API_KEY", "DECIMALAI_API_KEY")
ONLINE = False

if API_KEY:
    try:
        decimalai.init(api_key=API_KEY, langchain=True)
        ONLINE = True
    except Exception as exc:
        # init(verify=True) raises on 401/403 or an unreachable backend. An
        # expired key is not a reason to end the notebook in a traceback.
        print(f"!  that key was rejected: {type(exc).__name__}: {exc}")
        print("!  falling back to offline mode — every cell below still runs.\n")

if not ONLINE:
    decimalai.init(enabled=False)

# The LLM key is a different question. A DecimalAI key only decides whether runs
# are recorded; without an OpenAI key there is no agent run to record at all. Same
# three places, and exported to the environment because that is where the LLM
# client looks for it.
OPENAI_KEY = find_key("OPENAI_API_KEY")
if OPENAI_KEY.endswith("..."):
    OPENAI_KEY = ""  # the placeholder this cell used to assign, not a key
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
LLM_READY = bool(OPENAI_KEY)

print("DecimalAI: ONLINE  — LangChain runs will be traced to https://app.decimal.ai/traces"
      if ONLINE else
      "DecimalAI: OFFLINE — no usable key, so the SDK is initialized with tracing off.\n"
      "                     Nothing is captured; every cell below still runs.")
print("OpenAI:    ready — Steps 2-3 will call the model." if LLM_READY else
      "OpenAI:    missing — Steps 2-3 drive a real LLM, so they will print what they\n"
      "                     skipped instead of running. Set OPENAI_API_KEY (env var or\n"
      "                     Colab secret) and re-run this cell.")

## Step 2 — Build a Simple LangChain Agent

We'll create a tool-calling agent with a couple of tools using langchain 1.x's
`create_agent` (a LangGraph graph under the hood). DecimalAI captures
everything — you don't need to add any extra callbacks or wrappers.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool


# Define tools
@tool
def search_docs(query: str) -> str:
    """Search the knowledge base. Input: search query."""
    return f"Found 3 results for '{query}': [Article 1, Article 2, Article 3]"

@tool
def check_order(order_id: str) -> str:
    """Look up an order status. Input: order ID."""
    return f"Order {order_id}: Shipped on April 25, arriving April 29."

# Create agent. The tools above are plain Python and need no key; the model
# binding is the first thing that does, so that is what the guard is around.
agent = None

if LLM_READY:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    agent = create_agent(
        llm,
        [search_docs, check_order],
        system_prompt="You are a helpful customer support assistant.",
    )
    print("LangChain agent ready with 2 tools")
else:
    print("No OpenAI key, so no model to bind — the agent is not built.")
    print("The two tools are defined either way; set OPENAI_API_KEY and re-run")
    print("Step 1 and this cell to get the real thing.")

## Step 3 — Run Queries (Traces Are Auto-Captured)

In [ ]:
queries = [
    "How do I reset my password?",
    "Where is my order ORD-12345?",
    "What is your return policy?",
]

if agent is None:
    print("Skipped: this step sends the 3 queries above to a real LLM, which needs")
    print("an OpenAI key. Nothing else in the notebook depends on it.")
else:
    # Every invocation is automatically traced by DecimalAI
    for q in queries:
        print(f"\n{'='*50}")
        print(f"Q: {q}")
        result = agent.invoke({"messages": [{"role": "user", "content": q}]})
        print(f"A: {result['messages'][-1].content}")

    if ONLINE:
        decimalai.flush()  # drain the background sender before you go look
        print("\n3 traces auto-captured and sent to DecimalAI!")
        print("Open your dashboard: https://app.decimal.ai/traces")
    else:
        print("\n3 agent runs finished. DecimalAI is offline, so none of them were")
        print("captured — add a key in Step 1 to see them in the dashboard.")

## Step 4 — View in Dashboard

**Needs a DecimalAI key.** Open **[app.decimal.ai/traces](https://app.decimal.ai/traces)**.
For each trace you'll see:

- The full conversation flow (input → LLM → tool calls → output)
- Token usage and latency per LLM call
- Tool call inputs and outputs
- The auto-detected manifest (tools + model)

## Next Steps

- 📖 [Main Quickstart](./quickstart.ipynb) — See the version-aware manifest loop (runs with no keys at all)
- 📖 [Evaluations Guide](https://docs.decimal.ai/guides/evaluations) — Score your traces
- 📖 [Training Pipeline](https://docs.decimal.ai/tutorials/training-pipeline) — Build datasets from traces